In [1]:
import numpy as np
import pandas as pd

In [ ]:
def calc_burned_by_class(df, mtbs_class):
    """
    Sum burned area in hectares for a single MTBS severity class.

    Args:
        df (pd.DataFrame): Burned area records with columns mtbs_class,
            state_name, access, and area_km2.
        mtbs_class (int): MTBS severity class to filter on.

    Returns:
        pd.DataFrame: Columns state_name, access, and class_{mtbs_class}_ha.
    """
    # Keep only rows for this severity class, then total area per state/access pair
    grp = df[df['mtbs_class'] == mtbs_class].groupby(
        ['state_name', 'access'], as_index=False
    )['area_km2'].sum()

    # 1 km2 = 100 ha
    grp['area_ha'] = grp['area_km2'] * 100

    # Tag the area column with its class so repeated merges don't collide
    return grp[['state_name', 'access', 'area_ha']].rename(
        columns={'area_ha': f'class_{mtbs_class}_ha'}
    )


### File I/O

In [ ]:
# Load in the dataset
data_folder = "<PATH/TO/PROJECT/FOLDER>"


In [ ]:
# Read the CSV files
severity_df = pd.read_csv(f'{data_folder}/data/tables/mtbs_nfs_access_class_summary.csv')

# Read in the extent dataframe
extent_df = pd.read_csv(f'{data_folder}/data/tables/nfs_access_extent_km2.csv')
extent_df["extent_ha"] = extent_df["extent_masked_km2"] * 100
extent_df = extent_df[["state_name", "access_class", "extent_ha"]]


In [ ]:
# Filter for the full study period and severity classes 1-4
period_df = severity_df[
    severity_df['year'].between(1984, 2023) &
    severity_df['mtbs_class'].isin([1, 2, 3, 4])
].copy()


### Calculate burned area by state, access class, and severity class

In [ ]:
# Start from the extent table; standardize the join key name and copy so the
# original extent_df is left untouched by the merges below
summary = extent_df.rename(columns={'access_class': 'access'}).copy()

for cls in [1, 2, 3, 4]:
    # Left join keeps every state/access combination in the extent table,
    # including those with no burned area in this class
    summary = summary.merge(
        calc_burned_by_class(period_df, cls),
        on=['state_name', 'access'], how='left'
    )

# State/access pairs missing from a class group burned 0 ha, not unknown
summary = summary.fillna(0)


### Calculate percentages

In [ ]:
for cls in [1, 2, 3, 4]:
    summary[f'pct_{cls}'] = summary[f'class_{cls}_ha'] / summary['extent_ha'] * 100

summary['pct_all']           = summary[['pct_1', 'pct_2', 'pct_3', 'pct_4']].sum(axis=1)
summary['pct_low_mod_high']  = summary[['pct_2', 'pct_3', 'pct_4']].sum(axis=1)
summary['pct_mod_high']      = summary[['pct_3', 'pct_4']].sum(axis=1)



### Calculate ALL row (totals across all states)

In [ ]:
all_rows = []
for access in ['roaded', 'roadless', 'wilderness']:
    ext = extent_df[extent_df['access_class'] == access]['extent_ha'].sum()
    sub = summary[summary['access'] == access]
    row = {'state_name': 'ALL', 'access': access, 'extent_ha': ext}
    for cls in [1, 2, 3, 4]:
        row[f'class_{cls}_ha'] = sub[f'class_{cls}_ha'].sum()
        row[f'pct_{cls}'] = row[f'class_{cls}_ha'] / ext * 100 if ext > 0 else 0
    row['pct_all']          = sum(row[f'pct_{cls}'] for cls in [1, 2, 3, 4])
    row['pct_low_mod_high'] = sum(row[f'pct_{cls}'] for cls in [2, 3, 4])
    row['pct_mod_high']     = sum(row[f'pct_{cls}'] for cls in [3, 4])
    all_rows.append(row)

result = pd.concat([summary, pd.DataFrame(all_rows)], ignore_index=True)


### Format and display

In [ ]:
access_order  = ['roaded', 'roadless', 'wilderness']
access_labels = {'roaded': 'Developed', 'roadless': 'IRA', 'wilderness': 'Wilderness'}

state_order = sorted(result[result['state_name'] != 'ALL']['state_name'].unique())
state_rank  = {s: i for i, s in enumerate(state_order + ['ALL'])}
result['_state_rank']  = result['state_name'].map(state_rank)
result['_access_rank'] = result['access'].map({a: i for i, a in enumerate(access_order)})
result = result.sort_values(['_state_rank', '_access_rank']).reset_index(drop=True)

result['all_ha']          = result[['class_1_ha', 'class_2_ha', 'class_3_ha', 'class_4_ha']].sum(axis=1)
result['low_mod_high_ha'] = result[['class_2_ha', 'class_3_ha', 'class_4_ha']].sum(axis=1)
result['mod_high_ha']     = result[['class_3_ha', 'class_4_ha']].sum(axis=1)

def pct_fmt(pct):
    return f"{pct:.1f}%"

result_display = pd.DataFrame({
    'State':                        result['state_name'],
    'Access Class':                 result['access'].map(access_labels),
    'Forest Area (ha)':             result['extent_ha'].round(0).astype(int).apply(lambda x: f"{x:,}"),
    'Unburned to Low':              result['pct_1'].apply(pct_fmt),
    'Low':                          result['pct_2'].apply(pct_fmt),
    'Moderate':                     result['pct_3'].apply(pct_fmt),
    'High':                         result['pct_4'].apply(pct_fmt),
    'All Severity':                 result['pct_all'].apply(pct_fmt),
    'Low, Moderate, or High':       result['pct_low_mod_high'].apply(pct_fmt),
    'Moderate or High':             result['pct_mod_high'].apply(pct_fmt),
})

result_display


,State,Access Class,Forest Area (ha),Unburned to Low,Low,Moderate,High,All Severity,"Low, Moderate, or High",Moderate or High
0,Arizona,Developed,"1,943,897",11.5%,23.1%,9.3%,5.1%,49.0%,37.5%,14.4%
1,Arizona,IRA,"220,617",13.5%,27.3%,16.0%,6.6%,63.4%,49.9%,22.5%
2,Arizona,Wilderness,"241,322",11.1%,29.2%,25.3%,10.1%,75.6%,64.6%,35.4%
3,California,Developed,"4,301,027",6.7%,17.1%,15.2%,15.4%,54.4%,47.7%,30.7%
4,California,IRA,"887,139",11.7%,24.9%,22.7%,20.1%,79.4%,67.7%,42.8%
5,California,Wilderness,"1,120,003",16.8%,29.4%,22.3%,19.8%,88.4%,71.6%,42.2%
6,Colorado,Developed,"2,314,039",1.8%,3.0%,2.9%,3.0%,10.7%,8.9%,5.9%
7,Colorado,IRA,"1,285,555",1.6%,3.1%,3.7%,3.3%,11.7%,10.1%,7.0%
8,Colorado,Wilderness,"777,926",1.3%,2.6%,3.6%,3.8%,11.3%,10.0%,7.4%
9,Idaho,Developed,"2,520,875",4.0%,6.1%,4.9%,4.6%,19.5%,15.5%,9.5%


In [ ]:
result_display.to_csv(f"{data_folder}/data/tables/nfs_mtbs_severity_summary_1984_2023.csv", index=False)
